# B1.2 · Component summarisation and architecture synthesis

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

Builds on **[B1.1 · Historical parsing and structural indexing](https://spbreed.github.io/cyber-commons/lessons/B1.1.html)**.

| | |
|---|---|
| Open-source tooling | tree-sitter, Graphviz |
| Open-weight models | GLM-4.6, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Stages 1 and 2 produced units and their call relationships. That is still a
pile of functions. Phase 1 finishes by turning it into something a threat model
can be derived from.

**Stage 3 — Component summarisation.** Generate a localised summary per
directory or module: what it is for, what it talks to, what data passes through
it. Localised is the important word — summarising the whole repository at once
produces a paragraph that is true of every repository.

**Stage 4 — Architecture synthesis.** Compile those summaries into a single map
with three things on it:

- **entry points** — where untrusted input arrives,
- **data flows** — how it travels between components,
- **trust boundaries** — where it crosses from less trusted to more trusted.

The map is the artefact. Every later stage consumes it: threat modelling reads
the boundaries, planning allocates against them, feasibility filtering walks the
flows.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

> **About the model in this notebook.** It runs offline against a deterministic
> stand-in so the lesson executes on a Kaggle kernel with no network. The
> stand-in is not a language model and is labelled as such wherever it appears.
> To run the identical pipeline stage against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 2 · Stage 3 — summarise each component, locally

In [ ]:
import ast
from dataclasses import dataclass, field
from collections import defaultdict

SOURCES = {
 "src/web/handlers.py": '''
def get_report(request):
    """HTTP GET /reports/<id> — request.args is user-controlled."""
    return render(load_report(request.args["id"], request.args["owner"]))

def upload_doc(request):
    """HTTP POST /docs — multipart body is user-controlled."""
    return store(request.files["doc"], request.args["name"])
''',
 "src/data/reports.py": '''
def load_report(report_id, owner):
    return DB.execute("SELECT * FROM reports WHERE id=" + report_id +
                      " AND owner='" + owner + "'")
''',
 "src/data/docs.py": '''
def store(blob, name):
    path = "/srv/docs/" + name
    open(path, "wb").write(blob)
    return path
''',
 "src/util/render.py": '''
def render(rows):
    return "\\n".join(str(r) for r in rows)
''',
}

def units_of(src, path):
    tree = ast.parse(src)
    out = []
    for fn in [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]:
        calls = sorted({(c.func.id if isinstance(c.func, ast.Name)
                         else getattr(c.func, "attr", ""))
                        for c in ast.walk(fn) if isinstance(c, ast.Call)} - {""})
        doc = ast.get_docstring(fn) or ""
        out.append({"name": fn.name, "file": path, "params": [a.arg for a in fn.args.args],
                    "calls": calls, "doc": doc})
    return out

ALL_UNITS = [u for p, s in SOURCES.items() for u in units_of(s, p)]

DANGEROUS = {"execute": "database", "open": "filesystem", "write": "filesystem",
             "system": "shell", "get": "network"}

def summarise_component(directory, units):
    """Stage 3 — a LOCAL summary. Deterministic here; a model does this in production."""
    names = [u["name"] for u in units]
    external = sorted({c for u in units for c in u["calls"]
                       if c not in names and c in DANGEROUS})
    outbound = sorted({c for u in units for c in u["calls"] if c in
                       {x["name"] for x in ALL_UNITS} and c not in names})
    entry = [u["name"] for u in units if u["doc"].startswith("HTTP")]
    return {"component": directory, "units": names, "entry_points": entry,
            "talks_to": outbound,
            "touches": sorted({DANGEROUS[c] for c in external})}

by_dir = defaultdict(list)
for u in ALL_UNITS:
    by_dir["/".join(u["file"].split("/")[:-1])].append(u)

SUMMARIES = [summarise_component(d, us) for d, us in sorted(by_dir.items())]
for s in SUMMARIES:
    print(f"{s['component']}")
    print(f"   units       {s['units']}")
    print(f"   entry pts   {s['entry_points'] or '—'}")
    print(f"   talks to    {s['talks_to'] or '—'}")
    print(f"   touches     {s['touches'] or '—'}")
    print()

## 3 · Stage 4 — synthesise the architecture map

In [ ]:
def synthesise(summaries, units):
    by_name = {u["name"]: u for u in units}
    entry_points, flows, sinks = [], [], []
    for s in summaries:
        for e in s["entry_points"]:
            entry_points.append({"unit": e, "component": s["component"],
                                 "input": "HTTP request (untrusted)"})
    for u in units:
        for c in u["calls"]:
            if c in by_name:
                flows.append((u["name"], c))
            elif c in DANGEROUS:
                sinks.append({"unit": u["name"], "sink": c,
                              "resource": DANGEROUS[c]})
    return {"entry_points": entry_points, "flows": sorted(set(flows)), "sinks": sinks}

MAP = synthesise(SUMMARIES, ALL_UNITS)
print("ENTRY POINTS (untrusted input arrives here)")
for e in MAP["entry_points"]:
    print(f"   {e['unit']:14s} {e['component']:22s} {e['input']}")
print("\nDATA FLOWS")
for a, b in MAP["flows"]:
    print(f"   {a} → {b}")
print("\nSINKS (state changes / external resources)")
for s in MAP["sinks"]:
    print(f"   {s['unit']:14s} {s['sink']:10s} {s['resource']}")

## 4 · Trust boundaries — the part the map exists for

A boundary is any edge where data crosses from a less-trusted component into a more-trusted one. Those edges are where every finding in the rest of the pipeline will turn out to live.

In [ ]:
TRUST = {"src/web": 0, "src/data": 2, "src/util": 1}   # 0 = untrusted edge

def boundaries(flows, units):
    comp = {u["name"]: "/".join(u["file"].split("/")[:-1]) for u in units}
    out = []
    for a, b in flows:
        ca, cb = comp[a], comp[b]
        if TRUST.get(ca, 0) < TRUST.get(cb, 0):
            out.append({"edge": f"{a} → {b}", "from": ca, "to": cb,
                        "crossing": f"trust {TRUST[ca]} → {TRUST[cb]}"})
    return out

B = boundaries(MAP["flows"], ALL_UNITS)
print("TRUST BOUNDARY CROSSINGS")
for b in B:
    print(f"   {b['edge']:28s}{b['from']:10s} → {b['to']:10s} ({b['crossing']})")

reachable_sinks = []
entry_names = {e["unit"] for e in MAP["entry_points"]}
adj = defaultdict(list)
for a, b in MAP["flows"]: adj[a].append(b)
def walk(start, seen=None):
    seen = seen or set()
    if start in seen: return set()
    seen |= {start}
    out = {start}
    for n in adj[start]: out |= walk(n, seen)
    return out
for e in entry_names:
    for s in MAP["sinks"]:
        if s["unit"] in walk(e):
            reachable_sinks.append((e, s["unit"], s["resource"]))
print("\nSINKS REACHABLE FROM AN ENTRY POINT")
for e, u, res in reachable_sinks:
    print(f"   {e:14s} → {u:14s} touches {res}")
assert reachable_sinks

In [ ]:
# Verify: the map must change when the architecture changes.
SOURCES_V2 = dict(SOURCES)
SOURCES_V2["src/web/handlers.py"] = SOURCES["src/web/handlers.py"] + '''
def admin_export(request):
    """HTTP GET /admin/export — user-controlled, previously internal only."""
    return store(load_report(request.args["id"], request.args["owner"]),
                 request.args["name"])
'''
units_v2 = [u for p, s in SOURCES_V2.items() for u in units_of(s, p)]
by_dir2 = defaultdict(list)
for u in units_v2: by_dir2["/".join(u["file"].split("/")[:-1])].append(u)
map_v2 = synthesise([summarise_component(d, us) for d, us in sorted(by_dir2.items())],
                    units_v2)

before = {e["unit"] for e in MAP["entry_points"]}
after  = {e["unit"] for e in map_v2["entry_points"]}
print(f"entry points before: {sorted(before)}")
print(f"entry points after : {sorted(after)}")
print(f"NEW ENTRY POINT    : {sorted(after - before)}")
print(f"flows before {len(MAP['flows'])} → after {len(map_v2['flows'])}")
print("\nOne function added. A new untrusted entry point now reaches both the")
print("database and the filesystem. That delta is what B1.3 threat-models.")
assert after - before

## What you just proved

Three components summarise with their entry points, outbound calls and the resources they touch. The architecture map lists two HTTP entry points, the data flows between units, and three sinks. Two trust-boundary crossings are identified, both from `src/web` into `src/data`, and both database and filesystem sinks are reachable from an entry point. Adding one handler introduces a new entry point and extends the flow graph.

## Your turn

Draw the trust-boundary edges for one service you own. The interesting output is not the diagram — it is the count of sinks reachable from an untrusted entry point, which is the number Phase 3 will spend its budget on.

---

**Next → [B1.3 · Threat modelling from the architecture map](https://spbreed.github.io/cyber-commons/lessons/B1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*